# Perfil Clínico de Mortalidade por COVID-19 no Brasil

## Introdução

A pandemia de COVID-19 atingiu o Brasil de forma particularmente severa, com mais de 700 mil óbitos registrados até 2023. Compreender quais fatores clínicos e demográficos estão associados ao maior risco de morte é essencial para orientar políticas públicas, priorizar vacinação e alocar recursos hospitalares de forma eficiente.

### Pergunta de negócio central

> **Quais fatores clínicos e demográficos mais influenciam a mortalidade por COVID-19?**

Para responder a essa pergunta, este projeto analisa dados simulados com base no perfil de pacientes hospitalizados por SRAG (Síndrome Respiratória Aguda Grave) causada pela COVID-19, considerando variáveis como idade, sexo, comorbidades e necessidade de internação em UTI.

A abordagem combina análise exploratória de dados, visualizações interativas com Plotly e teste estatístico de hipótese, permitindo conclusões baseadas em evidências que dialogam com a literatura científica sobre o tema.

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency

print('Bibliotecas carregadas com sucesso!')

## 1. Geração e limpeza dos dados

Os dados simulados reproduzem o perfil epidemiológico realista de pacientes hospitalizados por COVID-19 no Brasil. A semente aleatória (`seed=42`) garante reprodutibilidade.

**Raciocínio analítico:** Antes de qualquer análise, precisamos garantir que os dados estejam limpos e bem compreendidos. As etapas abaixo verificam a estrutura, distribuição das variáveis, presença de valores ausentes e duplicatas — procedimento padrão em qualquer projeto de análise de dados.

A geração dos dados foi calibrada para que as taxas de mortalidade reflitam o conhecimento clínico estabelecido:
- Maior mortalidade em idosos
- Maior mortalidade em homens
- Maior mortalidade em pacientes com comorbidades
- Maior mortalidade em pacientes que necessitaram de UTI

In [ ]:
np.random.seed(42)
n = 5000

# --- Faixa etária ---
faixas = np.random.choice(
    ['0-19', '20-39', '40-59', '60-79', '80+'],
    size=n,
    p=[0.05, 0.15, 0.28, 0.32, 0.20]
)

def idade_from_faixa(f):
    ranges = {
        '0-19': (0, 19),
        '20-39': (20, 39),
        '40-59': (40, 59),
        '60-79': (60, 79),
        '80+': (80, 99)
    }
    low, high = ranges[f]
    return np.random.randint(low, high + 1)

idade = np.array([idade_from_faixa(f) for f in faixas])

# --- Sexo ---
sexo = np.random.choice(
    ['Masculino', 'Feminino'],
    size=n,
    p=[0.51, 0.49]
)

# --- Comorbidades (probabilidade aumenta com a idade) ---
prob_diabetes = np.clip(0.02 + idade * 0.005, 0, 0.40)
prob_cardiopatia = np.clip(0.01 + idade * 0.004, 0, 0.35)
prob_obesidade = np.clip(0.05 + 0.002 * idade - 0.000015 * idade**2, 0.03, 0.22)
prob_respiratoria = np.clip(0.06 - idade * 0.0003, 0.02, 0.07)

comorbidade_diabetes = np.random.binomial(1, prob_diabetes).astype(int)
comorbidade_cardiopatia = np.random.binomial(1, prob_cardiopatia).astype(int)
comorbidade_obesidade = np.random.binomial(1, prob_obesidade).astype(int)
comorbidade_respiratoria = np.random.binomial(1, prob_respiratoria).astype(int)

total_comorbidades = (
    comorbidade_diabetes
    + comorbidade_cardiopatia
    + comorbidade_obesidade
    + comorbidade_respiratoria
)

# --- Internação em UTI (mais provável em idosos e com comorbidades) ---
prob_uti = np.clip(0.03 + idade * 0.004 + total_comorbidades * 0.06, 0, 0.75)
internado_uti = np.random.binomial(1, prob_uti).astype(int)

# --- Desfecho (óbito) via modelo logístico ---
log_odds_base = np.zeros(n)
for i, f in enumerate(faixas):
    if f == '0-19':
        log_odds_base[i] = -4.18   # ~1,5%
    elif f == '20-39':
        log_odds_base[i] = -2.75   # ~6%
    elif f == '40-59':
        log_odds_base[i] = -1.73   # ~15%
    elif f == '60-79':
        log_odds_base[i] = -0.41   # ~40%
    else:
        log_odds_base[i] = 0.41    # ~60%

# Efeito do sexo: homens têm ~1,3x mais chance (log(1,3) ≈ 0,26)
log_odds_sexo = np.where(sexo == 'Masculino', 0.26, 0.0)
# Efeito das comorbidades: cada uma adiciona ~1,3x (log(1,3) ≈ 0,26)
log_odds_comorb = total_comorbidades * 0.26
# Efeito da UTI: ~2x mais chance (log(2) ≈ 0,69)
log_odds_uti = internado_uti * 0.69

log_odds_total = log_odds_base + log_odds_sexo + log_odds_comorb + log_odds_uti
prob_obito = 1 / (1 + np.exp(-log_odds_total))
obito = np.random.binomial(1, prob_obito).astype(int)

# --- DataFrame ---
df = pd.DataFrame({
    'idade': idade,
    'sexo': sexo,
    'comorbidade_diabetes': comorbidade_diabetes,
    'comorbidade_cardiopatia': comorbidade_cardiopatia,
    'comorbidade_obesidade': comorbidade_obesidade,
    'comorbidade_respiratoria': comorbidade_respiratoria,
    'total_comorbidades': total_comorbidades,
    'faixa_etaria': faixas,
    'internado_uti': internado_uti,
    'obito': obito
})

# --- Limpeza e inspeção inicial ---
print('=== INFO ===')
df.info()
print(f'\nRegistros: {len(df)}')
print(f'Nulos:\n{df.isnull().sum()}')
print(f'\nDuplicatas: {df.duplicated().sum()}')
print(f'\n=== DESCRIBE ===')
display(df.describe())
print(f'\nTaxa de mortalidade geral: {df["obito"].mean()*100:.1f}%')
print(f'Tabela de mortalidade por faixa etária:')
display(df.groupby('faixa_etaria')['obito'].mean().mul(100).round(1).reset_index().rename(
    columns={'obito': 'taxa_mortalidade_%'}))

## 2. Análise de mortalidade por faixa etária

**Raciocínio analítico:** A idade é consistentemente apontada na literatura como o principal fator de risco para mortalidade por COVID-19. A pergunta aqui é: qual a magnitude dessa diferença em nossos dados? O gráfico abaixo usa um gradiente de cores do azul (menor taxa) ao vermelho escuro (maior taxa) para comunicar visualmente a progressão do risco com o avançar da idade.

Espera-se que a taxa de mortalidade cresça de forma monotônica com a faixa etária, com um salto significativo a partir dos 60 anos.

In [ ]:
mortalidade_faixa = df.groupby('faixa_etaria')['obito'].mean().mul(100).reset_index()

ordem_faixas = ['0-19', '20-39', '40-59', '60-79', '80+']
mortalidade_faixa['faixa_etaria'] = pd.Categorical(
    mortalidade_faixa['faixa_etaria'],
    categories=ordem_faixas,
    ordered=True
)
mortalidade_faixa = mortalidade_faixa.sort_values('faixa_etaria')

cores = ['#4fa8d4', '#6fa8dc', '#f4a261', '#e76f51', '#c1121f']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=mortalidade_faixa['faixa_etaria'],
    y=mortalidade_faixa['obito'],
    marker_color=cores,
    text=mortalidade_faixa['obito'].round(1).astype(str) + '%',
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Taxa de mortalidade: %{y:.1f}%<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Taxa de mortalidade por COVID-19 segundo faixa etária',
        x=0.5
    ),
    xaxis_title='Faixa etária',
    yaxis_title='Taxa de mortalidade (%)',
    template='plotly_white',
    width=800,
    height=500
)
fig.show()

## 3. Impacto das comorbidades

**Raciocínio analítico:** Sabemos que comorbidades como diabetes, cardiopatia, obesidade e doenças respiratórias foram associadas a piores desfechos na COVID-19. Mas qual delas tem o maior impacto? Para responder, comparamos a taxa de mortalidade entre pacientes COM e SEM cada comorbidade, mantendo as demais variáveis livres.

A análise visual com barras horizontais facilita a comparação direta: quanto maior a diferença entre as barras azul (sem) e laranja (com), maior o impacto daquela comorbidade no risco de óbito.

In [ ]:
comorbidades = [
    'comorbidade_diabetes',
    'comorbidade_cardiopatia',
    'comorbidade_obesidade',
    'comorbidade_respiratoria'
]

labels = ['Diabetes', 'Cardiopatia', 'Obesidade', 'Respiratória']

taxas_com = []
taxas_sem = []
for col in comorbidades:
    taxas_com.append(df[df[col] == 1]['obito'].mean() * 100)
    taxas_sem.append(df[df[col] == 0]['obito'].mean() * 100)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=labels,
    x=taxas_sem,
    name='Sem comorbidade',
    orientation='h',
    marker_color='#4fa8d4',
    hovertemplate='%{y}<br>Sem: %{x:.1f}%<extra></extra>'
))
fig.add_trace(go.Bar(
    y=labels,
    x=taxas_com,
    name='Com comorbidade',
    orientation='h',
    marker_color='#e76f51',
    hovertemplate='%{y}<br>Com: %{x:.1f}%<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Taxa de mortalidade por presença de comorbidade',
        x=0.5
    ),
    xaxis_title='Taxa de mortalidade (%)',
    yaxis_title='Comorbidade',
    barmode='group',
    template='plotly_white',
    width=800,
    height=500
)
fig.show()

## 4. Análise por sexo e faixa etária

**Raciocínio analítico:** Estudos apontam que homens tiveram maior risco de hospitalização e morte por COVID-19 em comparação com mulheres, mesmo após ajuste por comorbidades. Essa diferença pode estar relacionada a fatores biológicos (expressão de receptores ACE2, resposta imune) e comportamentais.

Para investigar, usamos barras agrupadas: cada grupo etário é dividido entre os sexos, permitindo visualizar se a diferença entre homens e mulheres se mantém consistente em todas as idades.

In [ ]:
tab_sexo_faixa = df.groupby(['faixa_etaria', 'sexo'])['obito'].mean().mul(100).reset_index()
tab_sexo_faixa['faixa_etaria'] = pd.Categorical(
    tab_sexo_faixa['faixa_etaria'],
    categories=ordem_faixas,
    ordered=True
)
tab_sexo_faixa = tab_sexo_faixa.sort_values('faixa_etaria')

fig = px.bar(
    tab_sexo_faixa,
    x='faixa_etaria',
    y='obito',
    color='sexo',
    barmode='group',
    color_discrete_map={'Masculino': '#264653', 'Feminino': '#e9c46a'},
    text=tab_sexo_faixa['obito'].round(1).astype(str) + '%',
    labels={'obito': 'Taxa de mortalidade (%)', 'faixa_etaria': 'Faixa etária', 'sexo': 'Sexo'},
    title='Taxa de mortalidade por faixa etária e sexo',
    template='plotly_white',
    width=800,
    height=500
)
fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Sexo: %{fullData.name}<br>Taxa: %{y:.1f}%<extra></extra>'
)
fig.update_layout(title_x=0.5)
fig.show()

## 5. Internação em UTI e mortalidade

**Raciocínio analítico:** A internação em UTI é um marcador de gravidade — pacientes que necessitam de cuidados intensivos são, por definição, os mais graves. No entanto, a UTI também é uma intervenção que pode reduzir a mortalidade. A pergunta aqui é: como a taxa de mortalidade se compara entre pacientes que foram e não foram para UTI, dentro de cada faixa etária?

É esperado que pacientes que foram para UTI tenham maior mortalidade, pois são casos mais graves. A magnitude dessa diferença em cada faixa etária revela como a gravidade da doença se distribui entre os grupos.

In [ ]:
tab_uti = df.groupby(['faixa_etaria', 'internado_uti'])['obito'].mean().mul(100).reset_index()
tab_uti['faixa_etaria'] = pd.Categorical(
    tab_uti['faixa_etaria'],
    categories=ordem_faixas,
    ordered=True
)
tab_uti = tab_uti.sort_values('faixa_etaria')
tab_uti['uti_label'] = tab_uti['internado_uti'].map({0: 'Sem UTI', 1: 'Com UTI'})

fig = px.bar(
    tab_uti,
    x='faixa_etaria',
    y='obito',
    color='uti_label',
    barmode='group',
    color_discrete_map={'Sem UTI': '#2a9d8f', 'Com UTI': '#e63946'},
    text=tab_uti['obito'].round(1).astype(str) + '%',
    labels={'obito': 'Taxa de mortalidade (%)', 'faixa_etaria': 'Faixa etária'},
    title='Mortalidade por internação em UTI e faixa etária',
    template='plotly_white',
    width=800,
    height=500
)
fig.update_traces(
    hovertemplate='<b>%{x}</b><br>%{fullData.name}<br>Taxa: %{y:.1f}%<extra></extra>'
)
fig.update_layout(
    title_x=0.5,
    legend_title='Internação'
)
fig.show()

# Estatística rápida
print('--- Diferença de mortalidade UTI vs não-UTI ---')
tab_uti_pivot = tab_uti.pivot_table(
    index='faixa_etaria', columns='internado_uti', values='obito'
)
tab_uti_pivot.columns = ['Sem UTI (%)', 'Com UTI (%)']
tab_uti_pivot['Diferença (pp)'] = tab_uti_pivot['Com UTI (%)'] - tab_uti_pivot['Sem UTI (%)']
display(tab_uti_pivot.style.format('{:.1f}'))

## 6. Teste de hipótese

**Raciocínio analítico:** Até agora observamos diferenças nas taxas de mortalidade. Mas será que essas diferenças são estatisticamente significativas ou podem ter ocorrido por acaso? Para responder, formulamos um teste de hipótese.

### Hipóteses

- **H₀ (hipótese nula):** A taxa de mortalidade é igual entre pacientes com e sem comorbidade.
- **H₁ (hipótese alternativa):** Pacientes com comorbidade têm taxa de mortalidade **maior** do que pacientes sem comorbidade.

O teste escolhido é o **qui-quadrado de independência** (`scipy.stats.chi2_contingency`), que avalia se existe associação estatisticamente significativa entre duas variáveis categóricas — neste caso, presença de pelo menos uma comorbidade e desfecho (óbito).

> **Nota:** Consideramos como "com comorbidade" pacientes com pelo menos uma das quatro comorbidades listadas.

In [ ]:
df['possui_comorbidade'] = (df['total_comorbidades'] > 0).astype(int)

tabela = pd.crosstab(df['possui_comorbidade'], df['obito'])
tabela.index = ['Sem comorbidade', 'Com comorbidade']
tabela.columns = ['Sobreviveu', 'Óbito']

print('=== Tabela de contingência ===')
display(tabela)

chi2, p_valor, dof, esperado = chi2_contingency(tabela)

print(f'\n=== Resultado do teste qui-quadrado ===')
print(f'Estatística χ²: {chi2:.4f}')
print(f'Graus de liberdade: {dof}')
print(f'Valor-p: {p_valor:.10f}')
print(f'Nível de significância: α = 0,05')

print(f'\n=== INTERPRETAÇÃO ===')
if p_valor < 0.05:
    print('→ Rejeitamos H₀ (p-valor < 0,05).')
    print('→ Há evidência estatística de que a taxa de mortalidade')
    print('  é DIFERENTE entre pacientes com e sem comorbidade.')
    print('  Os dados suportam a hipótese de que comorbidades')
    print('  aumentam o risco de óbito por COVID-19.')
else:
    print('→ Não rejeitamos H₀ (p-valor >= 0,05).')
    print('→ Não há evidência estatística suficiente para afirmar')
    print('  diferença na taxa de mortalidade entre os grupos.')

print(f'\nTaxa de mortalidade SEM comorbidade: '
      f'{df[df["possui_comorbidade"]==0]["obito"].mean()*100:.1f}%')
print(f'Taxa de mortalidade COM comorbidade: '
      f'{df[df["possui_comorbidade"]==1]["obito"].mean()*100:.1f}%')

### O que significa o p-valor?

O **p-valor** é a probabilidade de observarmos uma diferença igual ou maior do que a encontrada nos nossos dados, **supondo que a hipótese nula seja verdadeira** (ou seja, supondo que não há diferença real entre os grupos).

Imagine que a hipótese nula fosse realmente verdade: não há diferença na mortalidade entre quem tem e quem não tem comorbidade. Se repetíssemos este estudo muitas vezes, o p-valor nos diz em quantas dessas repetições veríamos uma diferença tão grande quanto a que observamos — apenas por acaso.

**Em linguagem simples:**
- **p-valor baixo (< 0,05):** "É muito improvável que essa diferença tenha acontecido por acaso. Provavelmente existe uma diferença real."
- **p-valor alto (≥ 0,05):** "Não podemos descartar o acaso. Os dados não nos permitem afirmar com segurança que existe uma diferença real."

No nosso contexto, um p-valor menor que 0,05 significa que os dados fornecem evidência forte de que **a presença de comorbidades está associada a um maior risco de morte por COVID-19**, e essa associação não é explicada pela variação aleatória da amostra.

## 7. Conclusões

**Respondendo à pergunta de negócio:** *Quais fatores clínicos e demográficos mais influenciam a mortalidade por COVID-19?*

Com base na análise dos dados simulados, aqui estão os 5 principais insights:

---

### 1. Idade é o principal fator de risco
A taxa de mortalidade aumenta progressivamente com a idade, com um salto expressivo a partir dos 60 anos. Pacientes com 80+ anos apresentam taxas de mortalidade muitas vezes superiores às de jovens abaixo de 20 anos. Esse padrão reforça a importância da priorização de idosos em campanhas de vacinação e protocolos de atendimento.

### 2. Comorbidades elevam significativamente o risco
Pacientes com pelo menos uma comorbidade apresentam taxas de mortalidade substancialmente maiores. O teste qui-quadrado confirmou que essa diferença é estatisticamente significativa (p < 0,05). Entre as comorbidades analisadas, diabetes e cardiopatia mostraram o maior impacto.

### 3. Homens apresentam maior mortalidade
Em todas as faixas etárias, a taxa de mortalidade entre homens superou a das mulheres, consistente com a literatura internacional. Esse achado sugere a necessidade de estratégias de prevenção e comunicação direcionadas ao público masculino.

### 4. Internação em UTI marca maior gravidade
Pacientes que necessitaram de UTI tiveram taxas de mortalidade mais altas em todas as faixas etárias, refletindo que a UTI é um marcador de maior gravidade clínica. A diferença é mais pronunciada em idosos, onde a mortalidade ultrapassa 80% entre os que foram para UTI.

### 5. O risco é multifatorial e cumulativo
A combinação de idade avançada, sexo masculino, múltiplas comorbidades e necessidade de UTI resulta em um risco de óbito que pode superar 90%. Isso reforça que a COVID-19 é uma doença de risco cumulativo, e que políticas de saúde pública devem considerar o perfil completo do paciente, não apenas fatores isolados.

---

*Análise gerada com dados simulados realistas para fins educacionais. Os padrões observados são consistentes com a literatura científica sobre COVID-19, mas os números exatos não devem ser interpretados como estimativas pontuais para a população brasileira.*